# Module 2.2 -- The Model Context Protocol (MCP): Universal AI Tool Integration

**Track:** Backend Engineering for AI Applications  
**Toolchain:** mcp-python-sdk - JSON-RPC 2.0 - FastMCP  
**Objective:** Build MCP servers that expose enterprise tools to any LLM, and implement the advanced Sampling pattern for bidirectional LLM communication.

---

## The N-to-M Integration Problem

Without MCP, connecting N models to M tools requires N x M custom integrations:
```
GPT-4o  ----custom code----> PostgreSQL
GPT-4o  ----custom code----> Slack
Claude  ----custom code----> PostgreSQL  (DUPLICATE!)
Claude  ----custom code----> Slack       (DUPLICATE!)
Llama 3 ----custom code----> PostgreSQL  (TRIPLICATE!)
```
3 models x 2 tools = 6 integrations. Add one more tool? 3 more integrations.

With MCP, each tool is exposed ONCE as a standard server:
```
GPT-4o  --\                    /--> MCP Server: PostgreSQL
Claude  ----> MCP Protocol ----> MCP Server: Slack
Llama 3 --/                    \--> MCP Server: GitHub
```
3 models + 3 servers = 6 components (scales as N + M, not N x M).

### MCP Architecture

| Component | Role | Example |
|-----------|------|--------|
| **Host** | Application the user interacts with | Claude Desktop, VS Code, your app |
| **Client** | Routes messages between Host and Servers | Built into the Host |
| **Server** | Exposes tools/data via standard protocol | Your database connector |

### The Three MCP Primitives

| Primitive | Controlled By | Purpose | Analogy |
|-----------|--------------|---------|--------|
| **Resources** | Application | Inject context into LLM | GET endpoint (read-only) |
| **Tools** | Model (LLM) | Execute actions | POST endpoint (side effects) |
| **Prompts** | User | Reusable interaction templates | Saved macros |

### Transport Options

| Transport | Scope | Best For |
|-----------|-------|----------|
| **stdio** | Local process | Development, sidecar containers |
| **SSE/StreamableHTTP** | Remote HTTP | Microservices, enterprise |
| **WebSocket** | Bidirectional remote | High-throughput multi-agent |


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Every agent framework having its own custom tool system creates unmaintainable silos. Seniors adopt standard architectures like MCP, allowing Claude, OpenAI, and internal systems to universally connect to enterprise data.

**Mental Model:** MCP is USB-C for AI applications. It's a standard connector. Once your database or internal API exposes an MCP server, ANY compatible AI editor or agent can "plug in" and use it automatically.

**Common Junior Pitfall:** Hardcoding API keys and database credentials directly inside the prompt engineering logic, instead of abstracting it securely behind a standard tool server protocol like MCP.

---


---

## Syntax Bridge: JSON-RPC in 60 Seconds

MCP uses **JSON-RPC 2.0**, which is just a standard way to call functions remotely using JSON:

```json
// REQUEST: "Call the query_db function with this SQL"
{"jsonrpc": "2.0", "method": "tools/call", "params": {"name": "query_db", "arguments": {"sql": "SELECT *"}}, "id": 1}

// RESPONSE: "Here's the result"
{"jsonrpc": "2.0", "result": [{"type": "text", "text": "3 rows returned"}], "id": 1}
```

That's it. `method` = what to call. `params` = the arguments. `id` = to match request/response. Everything else in MCP is built on this simple pattern.

### Production Reality Check

| In This Notebook | In Production |
|---|---|
| Simulated tool discovery | Real server started via `python -m mcp.server.stdio` |
| Print statements | Actual JSON-RPC over stdio or HTTP |
| No auth | OAuth2 tokens, mTLS, credential vaults |
| Single server | Gateway routing to 10+ specialized MCP servers |


In [ ]:
# Cell 1 -- Install
!pip install -q mcp fastmcp pydantic


---
## 1 - Building an MCP Server

### How Does JSON-RPC 2.0 Work?

MCP uses JSON-RPC: a simple request/response protocol over any transport.
```json
// Request (Client -> Server)
{"jsonrpc": "2.0", "method": "tools/call", "params": {"name": "query_db", "arguments": {"sql": "SELECT * FROM users"}}, "id": 1}

// Response (Server -> Client)
{"jsonrpc": "2.0", "result": {"content": [{"type": "text", "text": "3 rows returned"}]}, "id": 1}
```


In [ ]:
# Cell 2 -- MCP Server implementation
#
# This defines a TOOL that an LLM can discover and call.
# The JSON schema description guides the LLM's reasoning about WHEN to use it.

mcp_server_code = '''
from mcp.server import Server
from mcp.types import Tool, TextContent
import json

server = Server("enterprise-db")

# Define a tool the LLM can call
@server.tool()
async def query_database(sql: str) -> list[TextContent]:
    """
    Execute a read-only SQL query against the enterprise database.
    Only SELECT statements are allowed. No mutations.
    Use this when the user asks about data, metrics, or records.
    """
    # Security: reject non-SELECT queries
    if not sql.strip().upper().startswith("SELECT"):
        return [TextContent(type="text", text="ERROR: Only SELECT queries allowed")]

    # In production: result = await db.execute(sql)
    result = {"columns": ["id", "name", "revenue"], "rows": [[1, "Acme", 52800000]]}
    return [TextContent(type="text", text=json.dumps(result, indent=2))]

# Define a resource (application-controlled context)
@server.resource("db://schema")
async def get_schema() -> str:
    """Return the database schema so the LLM can write valid SQL."""
    return """Tables:
    - users (id INT, name TEXT, email TEXT, created_at TIMESTAMP)
    - orders (id INT, user_id INT, amount DECIMAL, product TEXT)
    - metrics (date DATE, revenue DECIMAL, active_users INT)"""

# Run with: python -m mcp.server.stdio enterprise_db_server
'''

print('MCP Server Definition:')
print(mcp_server_code)
print('Security: Only SELECT queries accepted (least-privilege principle)')


In [ ]:
# Cell 3 -- Simulate MCP tool discovery and invocation
import json

# What the LLM "sees" when it discovers available tools
tool_manifest = {
    'tools': [
        {'name': 'query_database', 'description': 'Execute read-only SQL against enterprise DB. Only SELECT allowed.',
         'inputSchema': {'type': 'object', 'properties': {'sql': {'type': 'string', 'description': 'SQL SELECT query'}}, 'required': ['sql']}},
        {'name': 'search_documents', 'description': 'Semantic search across company knowledge base.',
         'inputSchema': {'type': 'object', 'properties': {'query': {'type': 'string'}, 'top_k': {'type': 'integer', 'default': 5}}, 'required': ['query']}},
    ]
}

print('MCP Tool Discovery (LLM sees this):')
for tool in tool_manifest['tools']:
    print(f'\n  Tool: {tool["name"]}')
    print(f'  Description: {tool["description"]}')
    print(f'  Parameters: {json.dumps(tool["inputSchema"]["properties"], indent=4)}')

# Simulate invocation
print(f'\n--- LLM decides to call query_database ---')
invoke = {'method': 'tools/call', 'params': {'name': 'query_database', 'arguments': {'sql': 'SELECT name, revenue FROM metrics WHERE date > 2026-01-01'}}}
print(f'Request: {json.dumps(invoke, indent=2)}')
response = {'result': {'content': [{'type': 'text', 'text': '{"rows": [["Q1", 52800000], ["Q2", 58100000]]}'}]}}
print(f'Response: {json.dumps(response, indent=2)}')


---
## 2 - The Sampling Pattern: Bidirectional LLM Communication

### What is Sampling?

Normally: Client asks Server to do something.  
With Sampling: **Server asks the LLM for help** during execution.

```
User: "Get Q3 revenue from the database"
  |-> Client sends to MCP Server
        |-> Server DOESN'T know the exact SQL syntax
        |-> Server uses Sampling: asks LLM "Generate SQL for Q3 revenue"
        |-> LLM returns: "SELECT SUM(revenue) FROM metrics WHERE quarter='Q3'"
        |-> Server validates and executes the SQL
        |-> Server returns result to Client
```

### Why This Matters

- Server keeps **security control** (validates SQL before execution)
- LLM provides **cognitive reasoning** (translates intent to SQL)
- Human stays **in the loop** (can approve sampling requests)


In [ ]:
# Cell 4 -- Sampling pattern implementation
sampling_code = '''
@server.tool()
async def smart_query(user_question: str) -> list[TextContent]:
    """
    Answer a natural language question about the database.
    Uses Sampling to ask the LLM to generate SQL.
    """
    # Step 1: Get schema context
    schema = await get_schema()

    # Step 2: SAMPLING -- ask the LLM to generate SQL
    sql_result = await server.request_sampling(
        messages=[{
            "role": "user",
            "content": f"Given this schema:\n{schema}\n\nGenerate a SELECT query for: {user_question}"
        }],
        system_prompt="You are a SQL expert. Return ONLY the SQL query, nothing else.",
        max_tokens=200,
    )

    generated_sql = sql_result.content.text

    # Step 3: VALIDATE (server keeps security control)
    if not generated_sql.strip().upper().startswith("SELECT"):
        return [TextContent(type="text", text="ERROR: LLM generated unsafe SQL")]

    # Step 4: Execute validated query
    result = await execute_query(generated_sql)
    return [TextContent(type="text", text=json.dumps(result))]
'''

print('Sampling Pattern (Server asks LLM for help):')
print(sampling_code)
print('Key: The SERVER validates the SQL before execution.')
print('     The LLM only provides cognitive assistance, not control.')


---
## Knowledge Check

### Q1: N+M vs NxM
You have 4 LLMs and 6 tools. How many integrations without MCP? With MCP?

<details><summary>Answer</summary>

Without MCP: 4 x 6 = 24 custom integrations. With MCP: 4 clients + 6 servers = 10 components. Adding 1 more tool: without MCP = +4 integrations (28 total). With MCP = +1 server (11 total).
</details>

### Q2: Resources vs Tools
When should you expose data as an MCP Resource vs a Tool?

<details><summary>Answer</summary>

**Resource**: Read-only context that the application injects (e.g., database schema, user profile). Controlled by the application. **Tool**: Actions with side effects that the LLM decides to invoke (e.g., executing SQL, sending email). Resources = GET, Tools = POST.
</details>

### Q3: Sampling Security
In the Sampling pattern, the server asks the LLM to generate SQL. What could go wrong?

<details><summary>Answer</summary>

The LLM might generate `DROP TABLE` or `DELETE FROM` statements. This is why the server MUST validate the generated SQL before execution (only allow SELECT). The Sampling pattern separates cognitive reasoning (LLM) from execution authority (server). Never trust LLM output without validation.
</details>


---
## Practical Practice

### Exercise 1: Design an MCP server for a weather API. Define 2 tools and 1 resource.
### Exercise 2: Write the JSON-RPC request/response for calling a `send_email` tool.


In [ ]:
# ===== SOLUTIONS =====
import json

print('Ex 1: Weather MCP Server Design')
server = {
    'name': 'weather-service',
    'tools': [
        {'name': 'get_forecast', 'description': 'Get weather forecast for a city',
         'params': {'city': 'string', 'days': 'integer (1-7)'}},
        {'name': 'get_alerts', 'description': 'Get active weather alerts for a region',
         'params': {'region': 'string'}},
    ],
    'resources': [
        {'uri': 'weather://supported-cities', 'description': 'List of cities with weather data'},
    ],
}
for t in server['tools']:
    print(f'  Tool: {t["name"]} - {t["description"]}')
for r in server['resources']:
    print(f'  Resource: {r["uri"]} - {r["description"]}')

print('\nEx 2: JSON-RPC for send_email')
request = {'jsonrpc': '2.0', 'method': 'tools/call',
           'params': {'name': 'send_email',
                      'arguments': {'to': 'john@example.com',
                                    'subject': 'Meeting Tomorrow',
                                    'body': 'Hi John, confirming our 2pm meeting.'}},
           'id': 42}
response = {'jsonrpc': '2.0',
            'result': {'content': [{'type': 'text', 'text': 'Email sent successfully'}]},
            'id': 42}
print(f'  Request:  {json.dumps(request, indent=2)}')
print(f'  Response: {json.dumps(response, indent=2)}')


---
## Summary

| Concept | What You Learned |
|---------|------------------|
| N-to-M problem | MCP reduces integrations from NxM to N+M |
| MCP primitives | Resources (context), Tools (actions), Prompts (templates) |
| JSON-RPC 2.0 | Standard request/response protocol |
| Sampling | Server asks LLM for cognitive help |
| Security | Credential isolation, least-privilege, SQL validation |

**Next -->** `13_monitoring_feedback.ipynb` -- Monitoring Drift & Feedback Loops